### Vectorless RAG with PageIndex

In [1]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")

In [5]:
from pageindex import PageIndexClient
from langchain_groq import ChatGroq

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = ChatGroq(model="qwen/qwen3.8-27b")

### Upload and Index PDF

In [ ]:
PDF_PATH = "./files/data_communication_computer_network_tutorial.pdf"

print("Uploading PDF to PageIndex...")
upload_res = pi_client.submit_document(PDF_PATH)
doc_id = upload_res["doc_id"]

print(f"Document uploaded with ID: {doc_id}")

Uploading PDF to PageIndex...
Document uploaded with ID: pi-cmu7b9xy1000h0ep9wwvyklbf.


### Build the tree index

In [7]:
print("Building the tree index ...")

doc_id = "pi-cmu7b9xy1000h0ep9wwvyklbf"

while True:
    status_res = pi_client.get_document(doc_id)
    status = status_res.get("status")

    if status == "completed":
        print("Indexing completed.")
        break
    elif status == "failed":
        print("Indexing failed.")
        break

    print("Indexing in progress...")
    time.sleep(5)


Building the tree index ...
Indexing completed.


### Inspect the tree structure

In [8]:
tree_res = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_res.get("result", [])

print(f"Tree index built with {len(pageindex_tree)} nodes.")
print("Raw tree:")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Tree index built with 32 nodes.
Raw tree:
{
  "title": "Preface",
  "node_id": "0000",
  "page_index": 1,
  "summary": "![img-0.jpeg](img-0.jpeg)\n\nLEARN DCN\n\ndata communication and computer network\n\ntutorialspoint\n\nSIMPLYEASYLEARNING\n\nwww.tutorialspoint.com\n\nhttps://www.facebook.com/tutorialspointindia\n\nhttps://twitter.com/tutorialspoint\n",
  "text": "![img-0.jpeg](img-0.jpeg)\n\nLEARN DCN\n\ndata communication and computer network\n\ntutorialspoint\n\nSIMPLYEASYLEARNING\n\nwww.tutorialspoint.com\n\nhttps://www.facebook.com/tutorialspointindia\n\nhttps://twitter.com/tutorialspoint\n"
}


### Retrieval

In Vector RAG retrieval
```query --> embed --> cosine_similarity(query_vec, all_chunk_vec) --> top-k chunks```

In PageIndex retrieval
```query + tree -> LLM reasons -> node 007 and 008 contains the answer```

In [ ]:
def llm_tree_search(query: str, tree: list) -> dict:

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n.get("title", ""),
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""
        You're given a query and a documents tree structure (like table of contents),
        Your task: Identify which node ids most likely contain the answer to the query.
        Think step by step about which sections are relevant.

        Query: {query}

        Document Tree:
        {json.dumps(compressed_tree, indent=2)}

        Reply only in this JSON format:
        {{
            "thinking": "Your reasoning about which nodes are relevant",
            "node_list":["node_id1", "node_id2", ...]
        }}    
    """
    response = groq_client.invoke(
        [{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.content)

# query = "Tell me about CSMA/CD and how it works in computer networks in short."
# res = llm_tree_search(query, pageindex_tree)

In [ ]:
# res

{'thinking': "CSMA/CD (Carrier Sense Multiple Access with Collision Detection) is a protocol used in Ethernet networks, which operates at the Data Link Layer (specifically the MAC sublayer). Therefore, the most relevant sections are those discussing the Data Link Layer. Node 0017 ('14. DATA LINK LAYER INTRODUCTION') is the primary candidate as it introduces the layer where this protocol resides. Node 0019 ('16. DATA LINK CONTROL AND PROTOCOLS') is also highly relevant as it discusses control mechanisms and protocols within the Data Link layer, which often covers access methods like CSMA/CD. While transmission media (Node 0014) like Ethernet cables are the physical medium where CSMA/CD is used, the protocol logic itself is defined in the Data Link Layer chapters.",
 'node_list': ['0017', '0019']}

In [23]:
def find_node_by_id(tree: list, target_ids: list) -> list:
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_node_by_id(node["nodes"], target_ids))
    return found

In [24]:
def generate_answer(query, nodes) -> str:
    if not nodes:
        return "No relevant nodes found to answer the query."
    
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page: {node.get("page_index", '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    
    context = "\n\n".join(context_parts)

    prompt = f"""
    You are a helpful assistant. Use the following context to answer the question:

    {context}

    Question: {query}
    """

    response = groq_client.invoke(
        [{"role": "user", "content": prompt}],
        response_format={"type": "text"}
    )
    return response.content

In [25]:
def vectorless_rag(query, tree, verbose=True) -> str:
    if verbose:
        print(f"{'=' * 50}")
        print(f"Query: {query}")
        print(f"{'=' * 50}")
    
    search_res = llm_tree_search(query, tree)
    node_ids = search_res.get("node_list", [])

    if verbose:
        print(f"\nReasoning: {search_res.get('thinking', 'No reasoning provided.')}")
        print(f"Identified Node IDs: {node_ids}")
    
    nodes = find_node_by_id(tree, node_ids)

    answer = generate_answer(query, nodes)
    return answer

In [26]:
query = "Tell me about CSMA/CD and how it works in computer networks in short."
answer = vectorless_rag(query, pageindex_tree, verbose=True)
print(f"Answer: {answer}")

Query: Tell me about CSMA/CD and how it works in computer networks in short.

Reasoning: CSMA/CD (Carrier Sense Multiple Access with Collision Detection) is a protocol used primarily in Ethernet networks, which operate at the Data Link Layer (and Physical Layer for the signaling). It is a medium access control (MAC) mechanism. Therefore, the most relevant sections would be those discussing the Data Link Layer, specifically protocols and control mechanisms, or potentially the Physical Layer if the book covers Ethernet standards there. Looking at the tree: 
1. Node 0017 (14. DATA LINK LAYER INTRODUCTION) is a strong candidate as it introduces the layer where MAC protocols reside.
2. Node 0019 (16. DATA LINK CONTROL AND PROTOCOLS) is the most likely location for specific protocols like CSMA/CD, as it discusses flow/error control and protocols in the data link layer.
3. Node 0011 (7. PHYSICAL LAYER INTRODUCTION) might contain references to signaling or Ethernet, but CSMA/CD is fundamentall